# 🏷️ Мониторинг цен конкурентов

**Среда выполнения → Выполнить всё**

Собирает ВСЕ товары с 3 сайтов, сопоставляет с твоими SKU, сохраняет в XLSX.

## Шаг 1: Установка

In [ ]:
# Python-пакеты + виртуальный дисплей
!pip install -q httpx lxml openpyxl playwright pyvirtualdisplay

# Системные библиотеки для Chromium
!apt-get update -qq 2>/dev/null
!playwright install-deps chromium 2>&1 | tail -1
!playwright install chromium 2>&1 | tail -1

print("Готово")

## Шаг 2: Загрузка проекта

In [ ]:
import os, shutil

# Всегда начинаем с корня
%cd /content

# Если проект уже есть — сохраняем my_skus.xlsx
if os.path.exists("price-monitor/monitor/__init__.py"):
    print("Проект уже загружен")
else:
    # Клонируем
    !git clone -q https://github.com/tswtim/price-monitor.git

%cd /content/price-monitor
!pwd
print("Готово")

## Шаг 3: Мои SKU

Если файла `my_skus.xlsx` нет — создастся шаблон. Загрузи свой файл сюда же и назови `my_skus.xlsx`.

In [ ]:
import os
if not os.path.exists("my_skus.xlsx"):
    !python -m monitor.cli --init-skus
    print("Создан шаблон my_skus.xlsx. Заполни его своими товарами и запусти заново.")
else:
    from openpyxl import load_workbook
    wb = load_workbook("my_skus.xlsx")
    count = wb.active.max_row - 1
    print(f"Найдено {count} SKU")

## Шаг 4: Запуск

Сбор всех товаров → сопоставление с SKU → result.xlsx

Для delikateska.ru нужен виртуальный дисплей — запускаем Xvfb.

In [ ]:
# Запускаем виртуальный дисплей (нужен для delikateska.ru)
from pyvirtualdisplay import Display
display = Display(visible=False, size=(1280, 1024))
display.start()
print("Дисплей запущен")

# Запуск мониторинга
!python -m monitor.cli --run

## Шаг 5: Результат

**Скачай `result.xlsx`** — слева в панели файлов, правый клик → Скачать.

### Листы в result.xlsx
- **Совпадения** — все продукты × SKU. Ставь `да`/`нет` в колонке `is_comparable`
- **Сводка по SKU** — мин/средняя/макс цена по каждому твоему товару
- **Мои SKU** — твои товары с ключевыми словами

### Как добавлять свои ссылки
Добавь строку в лист «Совпадения»: вставь URL, `matched_sku`, `is_comparable="да"`.
При следующем запуске скрипт сам сходит по ссылке и обновит цену.